RNAcentral's public PostgreSQL, read through the `rnacentral` Connection. The pod gets `CONN_RNACENTRAL_USER` and `CONN_RNACENTRAL_PASSWORD`; nothing secret is in this file. The two tables land in `bronze.public_db`.

In [ ]:
import os

import pandas as pd
import psycopg

conn = psycopg.connect(
    host="hh-pgsql-public.ebi.ac.uk", port=5432, dbname="pfmegrnargs",
    user=os.environ["CONN_RNACENTRAL_USER"],
    password=os.environ["CONN_RNACENTRAL_PASSWORD"],
    connect_timeout=20,
)
with conn.cursor() as cur:
    cur.execute("select id, descr, display_name, num_sequences, num_organisms, url from rnc_database order by id")
    databases = pd.DataFrame(cur.fetchall(), columns=["id", "descr", "display_name", "num_sequences", "num_organisms", "url"])
    cur.execute("select urs_taxid, taxid, rna_type, short_description from rnc_rna_precomputed limit 5000")
    sequences = pd.DataFrame(cur.fetchall(), columns=["urs_taxid", "taxid", "rna_type", "short_description"])
conn.close()
print(len(databases), "databases,", len(sequences), "sequences")


In [ ]:
# demo-reset drops every non-reserved schema in bronze nightly, so the pipeline owns its
# schema: create it first, or the create-table below is denied on a namespace Lakekeeper
# does not have.
%sql create schema if not exists bronze.public_db


In [ ]:
import stardelt

print(stardelt.write_df(databases, "bronze.public_db.rnacentral_databases", mode="overwrite"))
print(stardelt.write_df(sequences, "bronze.public_db.rnacentral_sequences", mode="overwrite"))
